In [ ]:
#import basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#libraries to standardise the columns and evaluate the models
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

#importing models to train the data
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [ ]:
#Loading dataset
data = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

#See first few rows
print("TOP ROWS OF THE DATA")
print(data.head())
print("\n")

# Check info and missing values
print("DATASET INFORMATION")
print(data.info())

print("\nMISSING VALUES")
print(data.isnull().sum())

print("\nDUPLICATES")
print(f"Duplicate rows: {data.duplicated().sum()}")

print("\nBASIC STATISTICS")
print(data.describe())

PREPROCESSING OF DATA

In [ ]:
#since there are no null values in any column we can skip that.

#there is one object datatype column - lifestyle activities
data['Lifestyle Activities'] = data['Lifestyle Activities'].map({'Yes': 1, 'No': 0})
test['Lifestyle Activities'] = test['Lifestyle Activities'].map({'Yes': 1, 'No': 0})

ANALYSING FEATURES


In [ ]:
#Plot all features
colors = ['coral', 'lightgreen', 'purple', 'orange']
columns = ['Therapy Hours', 'Initial Health Score', 'Average Sleep Hours', 'Follow-Up Sessions']
X_labels = ['Hours', 'Score', 'Hours', 'Sessions']

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

k=0
for i in range(0,2):
  for j in range (0,2):
  # Therapy Hours
    axes[i, j].hist(data[columns[k]], bins=20, edgecolor='black', color=colors[k], alpha=0.7)
    axes[i, j].set_title(columns[k], fontsize=12, fontweight='bold')
    axes[i, j].set_xlabel(X_labels[k])
    axes[i, j].grid(alpha=0.3)
    k+=1

# Lifestyle Activities
data['Lifestyle Activities'].value_counts().plot(kind='bar', ax=axes[2,0], color=['lightcoral', 'lightgreen'], edgecolor='black', alpha=0.7)
axes[2, 0].set_title('Lifestyle Activities', fontsize=12, fontweight='bold')
axes[2,0].set_xlabel('Activity')
axes[2,0].set_ylabel('Count')
axes[2,0].tick_params(axis='x', rotation=0)
axes[2,0].grid(alpha=0.3, axis='y')
axes[2,1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
correlation = data.corr()
print(correlation['Recovery Index'])

In [ ]:
#Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='RdYlGn', center=0,
            square=True, linewidths=1, fmt='.3f', cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()

print("CORRELATION WITH RECOVERY INDEX\n")
corr_with_target = correlation['Recovery Index'].sort_values(ascending=False)
for feature, corr_value in corr_with_target.items():
    if feature != 'Recovery Index':
        print(f"{feature:.<45} {corr_value:>6.3f}")


In [ ]:
# Scatter plots showing relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Therapy Hours vs Recovery
axes[0, 0].scatter(data['Therapy Hours'], data['Recovery Index'], alpha=0.4, s=20, color='coral')
axes[0, 0].set_xlabel('Therapy Hours', fontsize=11)
axes[0, 0].set_ylabel('Recovery Index', fontsize=11)
axes[0, 0].set_title('Therapy Hours vs Recovery Index', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
corr = data[['Therapy Hours', 'Recovery Index']].corr().iloc[0, 1]
axes[0, 0].text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=axes[0, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat'), fontsize=10, verticalalignment='top')

# Initial Health vs Recovery
axes[0, 1].scatter(data['Initial Health Score'], data['Recovery Index'], alpha=0.4, s=20, color='green')
axes[0, 1].set_xlabel('Initial Health Score', fontsize=11)
axes[0, 1].set_ylabel('Recovery Index', fontsize=11)
axes[0, 1].set_title('Initial Health Score vs Recovery Index', fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)
corr = data[['Initial Health Score', 'Recovery Index']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=axes[0, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat'), fontsize=10, verticalalignment='top')

# Sleep Hours vs Recovery
axes[1, 0].scatter(data['Average Sleep Hours'], data['Recovery Index'], alpha=0.4, s=20, color='purple')
axes[1, 0].set_xlabel('Average Sleep Hours', fontsize=11)
axes[1, 0].set_ylabel('Recovery Index', fontsize=11)
axes[1, 0].set_title('Sleep Hours vs Recovery Index', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
corr = data[['Average Sleep Hours', 'Recovery Index']].corr().iloc[0, 1]
axes[1, 0].text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat'), fontsize=10, verticalalignment='top')

# Follow-Up vs Recovery
axes[1, 1].scatter(data['Follow-Up Sessions'], data['Recovery Index'], alpha=0.4, s=20, color='orange')
axes[1, 1].set_xlabel('Follow-Up Sessions', fontsize=11)
axes[1, 1].set_ylabel('Recovery Index', fontsize=11)
axes[1, 1].set_title('Follow-Up Sessions vs Recovery Index', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)
corr = data[['Follow-Up Sessions', 'Recovery Index']].corr().iloc[0, 1]
axes[1, 1].text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=axes[1, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat'), fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
#drop the target feature and the feature least correlated with it -'Id'
X = data.drop(columns=['Recovery Index', 'Id']) #features to train on
y = data['Recovery Index']          #target

test_X = test.drop(columns=['Id'])

#using random_state = 0 throughout
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

#standardising-
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train),columns=X_train.columns,index=X_train.index)
X_val = pd.DataFrame(scaler.transform(X_val),columns=X_val.columns,index=X_val.index)
test_X = pd.DataFrame(scaler.transform(test_X),columns=test_X.columns,index=test_X.index)

print("-"*11)
print("DATA SPLIT")
print("-"*11)
print(f"Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")


In [ ]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=0),
    'Lasso': Lasso(alpha=1.0, random_state=0),
    'Decision Tree': DecisionTreeRegressor(random_state=0, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=0, max_depth=15),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=0, max_depth=5),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'XGB Regressor': XGBRegressor(n_estimators = 100,random_state=0,objective='reg:squarederror',n_jobs=-1)
}

#Train and evaluate
results = []

for name, model in models.items():
    #Training model on training data
    model.fit(X_train, y_train)

    #Testing on validation data
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    #evaluation
    val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    val_r2 = r2_score(y_val, y_pred_val)

    results.append({
        'Model': name,
        'Val RMSE': val_rmse,
        'Val R²': val_r2,
    })

    preds_test = model.predict(test_X)
    file_name = "normal_" + name.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".csv"

    pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_test}).to_csv(file_name, index=False)


#Create results DataFrame
results_df = pd.DataFrame(results).sort_values('Val RMSE')

print("BASELINE MODEL RESULTS (sorted by validation RMSE):\n")
print(results_df.to_string(index=False))

print("\nBEST MODEL:", results_df.iloc[0]['Model'])
print(f"Validation RMSE: {results_df.iloc[0]['Val RMSE']:.5f}")
print(f"Validation R²:   {results_df.iloc[0]['Val R²']:.5f}")

In [ ]:
#Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#RMSE comparison
axes[0].barh(results_df['Model'], results_df['Val RMSE'], color='skyblue', edgecolor='black')
axes[0].set_xlabel('Validation RMSE', fontsize=12)
axes[0].set_title('Model Comparison - RMSE', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(results_df['Val RMSE']):
    axes[0].text(v + 0.05, i, f'{v:.3f}', va='center', fontsize=10)

#R² comparison
axes[1].barh(results_df['Model'], results_df['Val R²'], color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Validation R²', fontsize=12)
axes[1].set_title('Model Comparison - R²', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(results_df['Val R²']):
    axes[1].text(v - 0.002, i, f'{v:.4f}', va='center', ha='right', fontsize=10)

plt.tight_layout()
plt.show()

***HYPERPARAMTER TUNING***

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)

linear_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

# Try different polynomial degrees
degrees = [1, 2, 3]
results = []

for d in degrees:
    linear_pipeline.set_params(poly__degree=d)
    scores = cross_val_score(
        linear_pipeline,
        X_train, y_train,
        cv=kf,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    mean_rmse = -scores.mean()
    results.append((d, mean_rmse))
    print(f"Degree {d} → Mean RMSE: {mean_rmse:.5f}")

best_degree, best_rmse = min(results, key=lambda x: x[1])
print(f"\nBest Polynomial Degree: {best_degree}")
print(f"Best RMSE: {best_rmse:.5f}\n")


In [ ]:
#using gridsearchCV

#Ridge
# Test different alpha values for Ridge
scoring = 'neg_root_mean_squared_error'
poly = PolynomialFeatures(degree=1, include_bias=False)


ridge_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=0))
])
param_grid_ridge = {
    'poly__degree': [1, 2, 3],            # try linear, quadratic, cubic
    'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100]
}


ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid=param_grid_ridge,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

ridge_grid.fit(X_train, y_train)

print("Best RIDGE parameters:", ridge_grid.best_params_)
print("Best RIDGE RMSE:", -ridge_grid.best_score_)

best_ridge = ridge_grid.best_estimator_
best_ridge.fit(X_train, y_train)
preds_ridge = best_ridge.predict(test_X)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_ridge}).to_csv("submission_ridge.csv", index=False)


In [ ]:
#Lasso
lasso_pipeline = Pipeline([
    ('poly', poly),
    ('scaler', scaler),
    ('lasso', Lasso(random_state=0, max_iter=1000))
])

param_grid_lasso = {
    'poly__degree': [1, 2, 3],
    'lasso__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    param_grid=param_grid_lasso,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

lasso_grid.fit(X_train, y_train)

print("Best LASSO parameters:", lasso_grid.best_params_)
print("Best LASSO RMSE:", -lasso_grid.best_score_)

best_lasso = lasso_grid.best_estimator_
best_lasso.fit(X_train, y_train)
preds_lasso = best_lasso.predict(test_X)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_lasso}).to_csv("submission_lasso.csv", index=False)

In [ ]:
#Random Forest

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=0))
])

param_grid_rf = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 5, 10],
    'rf__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid_rf,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

rf_grid.fit(X_train, y_train)

print("Best RF parameters:", rf_grid.best_params_)
print("Best RF RMSE:", -rf_grid.best_score_)

best_rf = rf_random.best_estimator_
best_rf.fit(X_train, y_train)
preds_rf = best_rf.predict(test_X)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_rf}).to_csv("submission_rf.csv", index=False)


In [ ]:
#KNeighbours
knn_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid_knn = {
    'poly__degree': [1, 2],
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance']
}

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid=param_grid_knn,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

knn_grid.fit(X_train, y_train)

print("Best KNN parameters:", knn_grid.best_params_)
print("Best KNN RMSE:", -knn_grid.best_score_)

best_knn = knn_grid.best_estimator_
best_knn.fit(X_train, y_train)
preds_knn = best_knn.predict(test_X)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_knn}).to_csv("submission_knn.csv", index=False)

In [ ]:
#XGB
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        random_state=0,
        n_jobs=-1,
        tree_method='hist'
    ))
])

param_dist_xgb = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgb__max_depth': [2, 3, 4, 5],
    'xgb__subsample': [0.7, 0.8, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_random = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=20,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=0
)


xgb_random.fit(X_train, y_train)
print("Best XGB parameters:", xgb_random.best_params_)
print("Best XGB RMSE:", -xgb_random.best_score_)

best_xgb = xgb_random.best_estimator_
best_xgb.fit(X_train, y_train)
preds_xgb = best_xgb.predict(test_X)
pd.DataFrame({"Id": test["Id"],"Recovery Index": preds_xgb}).to_csv("submission_xgb.csv", index=False)

In [ ]:
#Randomsied SearchCV for gradient boosting

gb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('gb', GradientBoostingRegressor(random_state=0))
])

param_dist_gb = {
    'gb__n_estimators': [100, 200, 300],
    'gb__learning_rate': [0.05,0.1,0.2],
    'gb__max_depth': [2, 3, 4],
    'gb__min_samples_split': [2, 5, 10]
}

gb_random = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist_gb,
    n_iter=20,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=0
)

gb_random.fit(X_train, y_train)

print("Best GB parameters:", gb_random.best_params_)
print("Best GB RMSE:", -gb_random.best_score_)

best_gb = gb_random.best_estimator_
best_gb.fit(X_train, y_train)
preds_gb = best_gb.predict(test_X)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_gb}).to_csv("submission_gb.csv", index=False)


***FEATURE ENGINEERING - WE TRY TO ADD/MODIFY FEATURES***

In [ ]:
# Create copies
X_new = X.copy()

X_new = pd.DataFrame(scaler.transform(X_new),columns=X_new.columns,index=X_new.index)
X_new['therapy_health_interaction'] = X_new['Therapy Hours'] * X_new['Initial Health Score']
X_new['sleep_health_interaction'] = (X_new['Average Sleep Hours'] * X_new['Initial Health Score'])

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y, test_size=0.2, random_state=0
)


test_new = test_X.copy();
test_new['therapy_health_interaction'] = test_new['Therapy Hours'] * test_new['Initial Health Score']
test_new['sleep_health_interaction'] = (test_new['Average Sleep Hours'] * test_new['Initial Health Score'])

In [ ]:
lin_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('lin', LinearRegression())
])

param_grid_lin = {'poly__degree': [1, 2, 3]}

lin_grid = GridSearchCV(
    lin_pipeline,
    param_grid=param_grid_lin,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

lin_grid.fit(X_train_new, y_train_new)

print("Best Linear params:", lin_grid.best_params_)
print("Best Linear RMSE:", -lin_grid.best_score_)

best_linear = lin_grid.best_estimator_
best_linear.fit(X_train_new, y_train_new)
preds_linear = best_linear.predict(test_new)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_linear}).to_csv("new_linear.csv", index=False)


#Ridge Regression
ridge_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=0))
])

param_grid_ridge = {
    'poly__degree': [1, 2, 3],
    'ridge__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid=param_grid_ridge,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

ridge_grid.fit(X_train_new, y_train)
print("\nBest Ridge params:", ridge_grid.best_params_)
print("Best Ridge RMSE:", -ridge_grid.best_score_)

best_ridge = ridge_grid.best_estimator_
best_ridge.fit(X_train_new, y_train_new)
preds_ridge = best_ridge.predict(test_new)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_ridge}).to_csv("new_ridge.csv", index=False)


#Lasso Regression
lasso_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('lasso', Lasso(random_state=0, max_iter=10000))
])

param_grid_lasso = {
    'poly__degree': [1, 2, 3],
    'lasso__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    param_grid=param_grid_lasso,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

lasso_grid.fit(X_train_new, y_train_new)
print("\nBest LASSO params:", lasso_grid.best_params_)
print("Best LASSO RMSE:", -lasso_grid.best_score_)

best_lasso = lasso_grid.best_estimator_
best_lasso.fit(X_train_new, y_train_new)
preds_lasso = best_lasso.predict(test_new)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_lasso}).to_csv("new_lasso.csv", index=False)


#Random Fores
rf_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=0))
])

param_dist_rf = {
    'poly__degree': [1, 2],
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [3, 5, 7, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist_rf,
    n_iter=10,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=0
)

rf_random.fit(X_train_new, y_train_new)
print("\nBest RF params:", rf_random.best_params_)
print("Best RF RMSE:", -rf_random.best_score_)

best_rf = rf_random.best_estimator_
best_rf.fit(X_train_new, y_train_new)
preds_rf = best_rf.predict(test_new)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_rf}).to_csv("new_rf.csv", index=False)

#Gradient Boosting (RandomizedSearch)
gb_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('gb', GradientBoostingRegressor(random_state=0))
])

param_dist_gb = {
    'poly__degree': [1, 2],
    'gb__n_estimators': [100, 200],
    'gb__learning_rate': np.linspace(0.05, 0.2, 5),
    'gb__max_depth': [2, 3, 4],
    'gb__min_samples_split': [2, 5]
}

gb_random = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist_gb,
    n_iter=10,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=0
)

gb_random.fit(X_train_new, y_train_new)
print("\nBest GB params:", gb_random.best_params_)
print("Best GB RMSE:", -gb_random.best_score_)

best_gb = gb_random.best_estimator_
best_gb.fit(X_train_new, y_train_new)
preds_gb = best_gb.predict(test_new)
pd.DataFrame({"Id": test["Id"], "Recovery Index": preds_gb}).to_csv("new_gb.csv", index=False)

Best Linear params: {'poly__degree': 1}
Best Linear RMSE: 2.03200052644115

Best Ridge params: {'poly__degree': 1, 'ridge__alpha': 0.01}
Best Ridge RMSE: 2.0320005262316667

Best LASSO params: {'lasso__alpha': 0.01, 'poly__degree': 1}
Best LASSO RMSE: 2.0312742855315955


In [ ]:
xgb_pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        random_state=0,
        n_jobs=-1,
        tree_method='hist'
    ))
])

param_dist_xgb = {
    'poly__degree': [1, 2],
    'xgb__n_estimators': [100, 200],
    'xgb__learning_rate': [0.05, 0.1, 0.2],
    'xgb__max_depth': [2, 3, 4],
    'xgb__subsample': [0.7, 0.8, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_random = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=0
)

xgb_random.fit(X_train_new, y_train_new)
print("\nBest XGB parameters:", xgb_random.best_params_)
print("Best XGB RMSE:", -xgb_random.best_score_)

best_xgb = xgb_random.best_estimator_
best_xgb.fit(X_train_new, y_train_new)
preds_xgb = best_xgb.predict(test_new)

pd.DataFrame({"Id": test["Id"],"Recovery Index": preds_xgb}).to_csv("new_xgb.csv", index=False)

In [ ]:
print("Weighted Averages\n")

# Try different weight combinations
weight_combos = [
    (0.7, 0.2, 0.1),  # Ridge heavy
    (0.6, 0.3, 0.1),
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.33, 0.33, 0.34)  # Equal weights
]

best_weights = None
best_ensemble_rmse = float('inf')

print("Testing different weight combinations...")
print("(Ridge, Random Forest, Gradient Boosting)")
print()

for weights in weight_combos:
    w1, w2, w3 = weights
    pred_weighted = w1 * best_ridge.predict(X_test_new) + w2 * best_lasso.predict(X_test_new) + w3 * best_linear.predict(X_test_new)
    rmse = np.sqrt(mean_squared_error(y_test_new, pred_weighted))

    print(f"Weights ({w1:.1f}, {w2:.1f}, {w3:.1f})  →  RMSE = {rmse:.4f}")

    if rmse < best_ensemble_rmse:
        best_ensemble_rmse = rmse
        best_weights = weights

print(f"\nBest weights: {best_weights}")
print(f"Best Ensemble RMSE: {best_ensemble_rmse:.4f}")